# Chyby, výjimky a logování

Chyby v Pythonu rozlišujeme na *syntaktické chyby* a *chyby za běhu* (*run-time*). Syntaktické chyby (nespárované závorky, špatné odsazení apod.) zabrání spuštění programu úplně. Run-time chyby vznikají nesprávným použitím funkce, chybějícími daty apod. — projeví se až při běhu.
<!-- TEASER_END -->

Chyby za běhu Python signalizuje mechanismem **výjimek** (*exceptions*), který si ukážeme v první části. Souběžně budeme používat knihovnu [**Loguru**](https://github.com/Delgan/loguru) pro **logování** — systematické zaznamenávání toho, co program dělá. Logování je od začátku lepší volba než `print()`: zprávy nesou čas, úroveň závažnosti i pozici v kódu, lze je filtrovat a ukládat do souboru. Loguru je jednodušší na použití než vestavěný `logging`.  Zároveň toho umí více (například [strukturované logování](https://github.com/Delgan/loguru/?tab=readme-ov-file#structured-logging-as-needed)).

In [1]:
import sys
from loguru import logger

Výjimka (*exception*) je vyhozena ve chvíli, kdy dojde k chybě. Pokud tuto výjimku nezachytíme (viz dále), běh programu se přeruší. Např. dělení nulou skončí výjimkou `ZeroDivisionError`:

In [2]:
1/0

ZeroDivisionError: division by zero

## Chytáme výjimky
Pokud nechceme, aby běh programu skončil ve chvíli výjimky, můžeme použít `try` - `except` blok. Ten funguje tak, že rizikovou část kód umístíme do `try` bloku, do `except` bloku pak umístíme instrukce pro případ chyby (výjimky).

In [3]:
try:
    1/0
except:
    logger.error("nulou se nedělí!")

2026-02-21 16:54:29.713 | ERROR    | __main__:<module>:4 - nulou se nedělí!


Takto použitý `except` odchytí všechny výjimky. To není obvykle dobrá praktika, je lepší odchytávat pouze určité výjimky (takové, které očekáváme, že mohou nastat). Pro odchycení dělení nulou by to vypadalo asi takto:

In [4]:
try:
    1/0
# do proměnné e se uloží vyhozená výjimka
except ZeroDivisionError as e:
    logger.error(f"nulou se nedělí! ({e})")

2026-02-21 16:54:30.352 | ERROR    | __main__:<module>:5 - nulou se nedělí! (division by zero)


Kompletní try-except blok může ještě obsahovat `else` a `finally` bloky, viz [dokumentace](http://docs.python.org/2/reference/compound_stmts.html#try). `finally` se hodí zejména pro "úklid", např. zavření souboru apod.

## Vyhazujeme výjimky
Výjimku můžeme samozřejmě vyhodit i v našem kódu pomocí klíčového slova `raise`. Pokud bychom chtěli např. kontrolovat vstup nějaké funkce, uděláme to takto:

In [5]:
def fact(n):
    """Spočítá faktoriál
    """
    from math import factorial
    try:
        # zkusíme převést n na číslo
        n = float(n)
    except ValueError as e:
        raise ValueError("{} není číslo".format(n))
    # je n celé číslo >= 0
    if not n.is_integer() or n < 0:
        raise ValueError("{} není celé číslo >= 0".format(n))
    else:
        n = int(n)
    return factorial(n)

# jednoduchý test
n = 4
print(f"{n}! = {fact(n)}")

4! = 24


In [6]:
# teď zkusíme, jestli fungují naše výjimky
for n in (4.0, "0", 3.2, -1):
    try:
        print(f"{n}! = {fact(n)}")
    # Exception odchytí v podstatě jakoukoli výjimku, ostatní výjimky totiž od ní dědí
    except Exception as e:
        logger.warning(str(e))

2026-02-21 16:54:32.457 | WARNING  | __main__:<module>:7 - 3.2 není celé číslo >= 0
2026-02-21 16:54:32.457 | WARNING  | __main__:<module>:7 - -1.0 není celé číslo >= 0


4.0! = 24
0! = 1


## Traceback a `logger.exception()`

Chceme-li v bloku `except` zalogovat celý traceback výjimky, použijeme `logger.exception()`. Je to ekvivalent `logger.error()`, který navíc automaticky připojí traceback aktuální výjimky — bez nutnosti importovat modul `traceback`:

In [7]:
try:
    fact(1.1)
except Exception:
    # logger.exception() = logger.error() + automatický traceback aktuální výjimky
    logger.exception("Zachycena výjimka:")

2026-02-21 16:54:34.456 | ERROR    | __main__:<module>:5 - Zachycena výjimka:
Traceback (most recent call last):

  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/jakub/workspace/fjfi/python-fjfi/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
    │   └ <bound method Application.launch_instance of <class 'ipykernel.kernelapp.IPKernelApp'>>
    └ <module 'ipykernel.kernelapp' from '/Users/jakub/workspace/fjfi/python-fjfi/.venv/lib/python3.12/site-packages/ipykernel/kern...
  File "/Users/jakub/workspace/fjfi/python-fjfi/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
    │   └ <function IPKernelApp.start at 0x10c3ff2e0>
    └ <ipykernel.kernelapp.IPKernelApp object at 0x1057a1e20>
  File "/Users/jakub/workspace/fjfi/python-fjfi/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", lin

## Logování do souboru

Logy z dlouhých výpočtů chceme uchovávat v souboru. Loguru to umožňuje jednoduše přidáním dalšího *sink* (výstupu). Zároveň lze nastavit automatické otáčení souboru (*rotation*) a mazání starých záznamů (*retention*).

In [8]:
# Přidáme výstup do souboru
# rotation="10 MB"  -- vytvoří nový soubor, jakmile aktuální přesáhne 10 MB
# retention="7 days" -- automaticky smaže záznamy starší 7 dní
logger.add(
    "vypocet.log",
    level="DEBUG",
    rotation="10 MB",
    retention="7 days",
    encoding="utf-8",
)

logger.info("Spouštím výpočet...")
for i in range(3):
    logger.debug(f"Iterace {i}")
logger.success("Výpočet dokončen.")

2026-02-21 16:54:37.074 | INFO     | __main__:<module>:12 - Spouštím výpočet...
2026-02-21 16:54:37.074 | DEBUG    | __main__:<module>:14 - Iterace 0
2026-02-21 16:54:37.075 | DEBUG    | __main__:<module>:14 - Iterace 1
2026-02-21 16:54:37.075 | DEBUG    | __main__:<module>:14 - Iterace 2
2026-02-21 16:54:37.076 | SUCCESS  | __main__:<module>:15 - Výpočet dokončen.


In [9]:

# Zobrazíme obsah souboru
with open("vypocet.log", encoding="utf-8") as f:
    print(f.read())

2026-02-21 16:54:37.074 | INFO     | __main__:<module>:12 - Spouštím výpočet...
2026-02-21 16:54:37.074 | DEBUG    | __main__:<module>:14 - Iterace 0
2026-02-21 16:54:37.075 | DEBUG    | __main__:<module>:14 - Iterace 1
2026-02-21 16:54:37.075 | DEBUG    | __main__:<module>:14 - Iterace 2
2026-02-21 16:54:37.076 | SUCCESS  | __main__:<module>:15 - Výpočet dokončen.



*Pozor: `logger.add` přidá vždy další výstup. Pokud chceme výstupy nejprve odstranit, použijeme `logger.remove()`.*

## Doporučení pro vědecké skripty

- Nahraďte `print()` za `logger.info()` / `logger.debug()`.
- Logujte do souboru při dlouhých výpočtech (neztrácejte výstupy).
- Používejte `@logger.catch` nebo `logger.exception()` místo ručního vypisování tracebacků.